# 05 — Chronic Condition Burden Analysis

Analyzes the relationship between chronic condition comorbidity and healthcare utilization/costs.

Focuses on: multi-condition patterns, spending escalation with comorbidity, and condition correlations.

Requires `cms_data.db` from `01_setup.ipynb`.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
DB = Path("../cms_data.db")
conn = sqlite3.connect(DB)

---
## 1. Spending escalation with number of chronic conditions

In [ ]:
bene = pd.read_sql("""
SELECT
  DESYNPUF_ID,
  YEAR,
  MEDREIMB_IP + MEDREIMB_OP + MEDREIMB_CAR as total_spend,
  SP_ALZHDMTA, SP_CHF, SP_CHRNKIDN, SP_CNCR, SP_COPD, SP_DEPRESSN,
  SP_DIABETES, SP_ISCHMCHT, SP_OSTEOPRS, SP_RA_OA, SP_STRKETIA
FROM beneficiary_summary
""", conn)

# Count conditions per beneficiary (1 = has condition)
chronic_cols = ['SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN',
                'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA']

bene['num_conditions'] = (bene[chronic_cols] == 1).sum(axis=1)

# Spending by condition burden
condition_burden = bene.groupby('num_conditions').agg(
    num_beneficiaries=('DESYNPUF_ID', 'count'),
    avg_spend=('total_spend', 'mean'),
    median_spend=('total_spend', 'median'),
    max_spend=('total_spend', 'max'),
    pct_90=('total_spend', lambda x: x.quantile(0.90))
).reset_index()

condition_burden = condition_burden.round(0)

print("Spending by Number of Chronic Conditions")
print("="*90)
display(condition_burden)
condition_burden.to_csv("../data/exports/spending_by_condition_burden.csv", index=False)

---
## 2. Condition co-occurrence (comorbidity patterns)

In [ ]:
# For beneficiaries with each condition, what % also have other conditions?
comorbidity = pd.DataFrame()

for condition in chronic_cols:
    with_cond = bene[bene[condition] == 1]
    if len(with_cond) == 0:
        continue
    
    row = {'primary_condition': condition}
    for other_cond in chronic_cols:
        if condition != other_cond:
            pct_also_have = (with_cond[other_cond] == 1).mean() * 100
            row[f"{other_cond}_pct"] = round(pct_also_have, 1)
    
    comorbidity = pd.concat([comorbidity, pd.DataFrame([row])], ignore_index=True)

print("\nComorbidity Patterns: If patient has Condition X, likelihood of also having Y")
print("(showing top 3 co-conditions for each primary condition)")
print("="*90)

for idx, row in comorbidity.iterrows():
    primary = row['primary_condition']
    co_cols = [c for c in row.index if c.endswith('_pct')]
    co_values = [(c.replace('_pct', ''), row[c]) for c in co_cols if pd.notna(row[c])]
    co_values.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n{primary}:")
    for cond, pct in co_values[:3]:
        print(f"  + {cond}: {pct:.1f}%")

---
## 3. High-risk comorbidity combinations

In [ ]:
# Spending for specific dual-condition combinations
dual_combinations = pd.DataFrame()

combinations = [
    ('SP_DIABETES', 'SP_CHF'),
    ('SP_DIABETES', 'SP_ISCHMCHT'),
    ('SP_CHF', 'SP_COPD'),
    ('SP_ISCHMCHT', 'SP_DEPRESSN'),
    ('SP_CHRNKIDN', 'SP_DIABETES'),
]

for cond1, cond2 in combinations:
    both = bene[(bene[cond1] == 1) & (bene[cond2] == 1)]
    either = bene[((bene[cond1] == 1) | (bene[cond2] == 1))]
    neither = bene[(bene[cond1] == 2) & (bene[cond2] == 2)]
    
    row = {
        'combination': f"{cond1.replace('SP_', '')} + {cond2.replace('SP_', '')}",
        'both_conditions_avg_spend': round(both['total_spend'].mean()) if len(both) > 0 else None,
        'either_condition_avg_spend': round(either['total_spend'].mean()) if len(either) > 0 else None,
        'neither_condition_avg_spend': round(neither['total_spend'].mean()) if len(neither) > 0 else None,
        'prevalence_both_pct': round((len(both) / len(bene) * 100), 1)
    }
    dual_combinations = pd.concat([dual_combinations, pd.DataFrame([row])], ignore_index=True)

print("\nSpending Impact of Dual-Condition Combinations")
print("="*90)
display(dual_combinations)
dual_combinations.to_csv("../data/exports/dual_condition_combinations.csv", index=False)

---
## 4. Condition progression by year

In [ ]:
# For each condition, what % of beneficiaries developed it year-over-year?
bene_pivot = bene.pivot_table(
    index='DESYNPUF_ID',
    columns='YEAR',
    values=chronic_cols[0],
    aggfunc='first'
)

progression = pd.DataFrame()

for condition in chronic_cols:
    data_2008 = bene[bene['YEAR'] == 2008][condition].value_counts()
    data_2009 = bene[bene['YEAR'] == 2009][condition].value_counts()
    data_2010 = bene[bene['YEAR'] == 2010][condition].value_counts()
    
    pct_2008 = (data_2008.get(1, 0) / len(bene[bene['YEAR'] == 2008]) * 100).round(1) if len(bene[bene['YEAR'] == 2008]) > 0 else 0
    pct_2009 = (data_2009.get(1, 0) / len(bene[bene['YEAR'] == 2009]) * 100).round(1) if len(bene[bene['YEAR'] == 2009]) > 0 else 0
    pct_2010 = (data_2010.get(1, 0) / len(bene[bene['YEAR'] == 2010]) * 100).round(1) if len(bene[bene['YEAR'] == 2010]) > 0 else 0
    
    row = {
        'condition': condition,
        'prevalence_2008': pct_2008,
        'prevalence_2009': pct_2009,
        'prevalence_2010': pct_2010,
        'change_2008_to_2010': (pct_2010 - pct_2008).round(1)
    }
    progression = pd.concat([progression, pd.DataFrame([row])], ignore_index=True)

print("\nCondition Prevalence Over Time (2008–2010)")
print("="*90)
display(progression)
progression.to_csv("../data/exports/condition_progression_by_year.csv", index=False)

In [ ]:
conn.close()
print("\n✓ Chronic condition analysis complete. Outputs saved to data/exports/")